# Day 6 — Evaluation & ablations
Fair-comparison rules: same held-out clips for every system; fixed scoring normalization (`src.evaluation.scoring_normalize`);
rules not tuned on test; raw and post-processed reported separately; regressions reported.

In [ ]:
# --- Setup (run once per Colab session) ---
REPO_URL = "https://github.com/arkalatua/Indic_Switch.git"
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.exists("/content/Indic_Switch"):
        subprocess.run(["git", "clone", REPO_URL, "/content/Indic_Switch"], check=True)
    %cd /content/Indic_Switch
    !pip install -q -r requirements.txt
    # Optional: persist models/results across sessions
    # from google.colab import drive; drive.mount("/content/drive")
else:
    os.chdir(os.path.abspath(".."))  # running locally from notebooks/
sys.path.insert(0, os.getcwd())

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

In [ ]:
import glob
preds = " ".join(sorted(glob.glob("experiments/asr/*.jsonl")))
lids = " ".join(f"--lid {m}" for m in ["models/lid_model", "models/ngram_lid.pkl", "rule"]
                if glob.glob(m + "*") or m == "rule")
!python scripts/run_experiments.py {preds} {lids}

In [ ]:
import pandas as pd, json
df = pd.read_csv("experiments/results.csv")
display(df.round(4))
abl = json.load(open("experiments/ablations.json"))

## Switch-density breakdown and examples (improved *and* degraded)

In [ ]:
key = next(k for k in abl if k.endswith("+full"))
print(key); display(pd.DataFrame({b: {"n": v["n"], "raw_WER": v["raw"]["wer"], "sys_WER": v["system"]["wer"]}
                                   for b, v in abl[key]["by_switch_density"].items()}).T)
for ex in abl[key]["examples"]:
    print(f"Δ={ex['delta_wer']:+.3f}\n REF : {ex['ref']}\n RAW : {ex['raw']}\n NORM: {ex['normalized']}\n")

## LID metrics summary (E3) and efficiency

In [ ]:
for name in ["script_rule", "char_ngram_lr", "xlmr"]:
    try: r = json.load(open(f"experiments/lid_{name}_test.json"))
    except FileNotFoundError: continue
    print(f"{name:14s} macro-F1 {r['lid']['macro_f1']:.4f} boundary-F1 {r['boundary']['f1']:.4f} "
          f"{r['efficiency']['tokens_per_second']:.0f} tok/s on {r['efficiency']['device']}")

## Write-up checklist
- [ ] Hardware (GPU model), batch size, whether model loading is in RTF
- [ ] Per-system WER/CER (exact + script-agnostic), LID macro-F1, boundary F1
- [ ] Ablations: −LID (rule tagger), −canonicalize, −loop collapse, script conversion, prompting
- [ ] Degraded cases discussed honestly